#Clustering
In this notebook, we conducted a comprehensive analysis using clustering methodologies of a medical dataset aimed at developing a question-answering model. Our analysis involved utilizing preprocessed data and conducting clustering and visualization techniques to gain insights into the structure and content of the dataset.



*   **Data Exploration and Preprocessing:**
We began by exploring the dataset's characteristics, including word counts, vocabulary richness, and distribution of words. Preprocessing steps such as removing punctuation, stopwords, and lemmatization were applied to standardize the text and improve model performance.
*   **Visualization and Analysis:**
We visualized the dataset using techniques like WordClouds, N-grams, and TF-IDF analysis to uncover common phrases, patterns, and significant terms. This allowed us to gain a deeper understanding of the dataset's content and identify prevalent medical themes.

# Import and install

In [ ]:
TEST_MODEL = True

In [ ]:
if TEST_MODEL:
  !git clone https://github.com/epfml/sent2vec.git
  %cd sent2vec
  !make
  !pip install .
  import sent2vec

In [ ]:
!pip install datasets

In [ ]:
import string
from datasets import load_dataset
import re
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import LabelEncoder
import nltk
from nltk.corpus import stopwords
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import altair as alt
import seaborn as sns
from gensim.models import Word2Vec
from google.colab import drive
import os
import csv
import numpy as np
import random
from nltk.stem import WordNetLemmatizer
from sklearn.cluster import KMeans
import time
from sklearn import metrics
from sklearn.cluster import KMeans
import time
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

path = 'Colab Notebooks/NLP'

os.chdir(f'/content/drive/MyDrive/{path}')
os.getcwd()

In [ ]:
DATAS_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Datasets'

#Dataset import

In [ ]:
# For the clustering module we have used the pre-processed dataset from our preliminary analysis which has all the details such as
# input and output length, vocabulary and lemmatization
file_dataframe = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP/NLP Project/Datasets/pre_preprocessed_dataset.csv')
print(type(file_dataframe))

In [ ]:
# The below is out preprocessed dataframe

file_dataframe.head()

In [ ]:
input, output = file_dataframe["input"], file_dataframe["output"]

In [ ]:
documents = input + " " + output

# Documents vectorisation

In order to cluster the documents, we need to first convert them into a vector format. We will use the `TfidfVectorizer` from Scikit-Learn to do this.

The `TfidfVectorizer` is very similar to the `CountVectorizer` we used in the text classification tutorial except that it multiplies the term frequency in the document by the inverse document frequecy of the term across the corpus: $\mathrm{tf}(t, d) \cdot \mathrm{idf}(t)$
- here $\mathrm{tf}(t, d)$ is the count of the term $t$ in the document $d$
- idf is inverse document frequency: $\mathrm{idf}(t) = \log{\frac{n + 1}{\mathrm{df}(t) + 1}} + 1$
- $n$ is the number of documents in collection
- and the document frequency, $\mathrm{df}(t)$ is the number of documents that contain the term $t$
- see https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html

The use of idf-weighting can be justified in term of information theory:
- the amount of **information** associated with a term quantifies the **amount of surprise** at seeing a term in a document
- the surprise decreases with its (prior) probability of occurrence and information must be additive, so we have:
  - $I(t) = \log{\frac{1}{P(t)}} = -\log{P(t)}$
- the probability of observing a particular term in a document is just the percentage of documents that contain the term:
  - $P(t) = \frac{\mathrm{df}(t)}{n}$
- We smooth this estimate so that small df(t) values don't cause unreasonably high idf values, we have:
  - $P(t) = \frac{\mathrm{df}(t)+1}{n+1}$
  - so $I(t) = \log{\frac{n+1}{\mathrm{df}(t)+1}}$
- The idf value used is then $\mathrm{idf}(t) = I(t) + 1$, where the $+1$ prevents idf from going to zero as df approaches $n$.

In [ ]:
vectorizer = TfidfVectorizer(max_df=0.8, min_df=5, stop_words='english', )

The vectorisation method takes a number of arguments that control the resulting vocabulary. We have set the following arguments:  
- **max_df = 0.8**: remove words occuring in more than half of the documents (note: this will get rid of any corpus-specific tags)
- **min_df = 5**: remove words occuring in less than 5 documents  
- **stop_words = 'english'**: remove stopwords using an english stopword list

We can now fit the vectorizer to the data:
- Note: we could transform the data at the same time, using the `fit_transform()` method, but we'll wait for now to transform the data

In [ ]:
vectorizer.fit(documents)

Let's have a quick look at the vocabulary. How big is it?

In [ ]:
vocab = vectorizer.get_feature_names_out()

print(f"Length of vocabulary: {len(vocab)}")

Wow, that's quite big!

Let's print out a random sample of 100 terms from it to see what they look like:

In [ ]:
sorted(random.sample(vocab.tolist(),100))

Most of the terms look pretty good, but the representation looks reasonable.

Now let's vectorize the dataset:

In [ ]:
vector_outputs = vectorizer.transform(output)

Here is the sparse vector for the first document:


In [ ]:
print(vector_outputs[0])

The decimial values are the TF-IDF scores for the terms. We can sort the terms by their TF-IDF values, and print them out as follows:

In [ ]:
sorted([(vocab[j], vector_outputs[0, j]) for j in vector_outputs[0].nonzero()[1]], key=lambda x: -x[1])

Do the top terms agree with what you expected for the document?

Print it out below:

In [ ]:
print(output[0])

## Data exploration

Let's play a bit with our data

### Measuring the similarity between vectorised documents

The vectorizer also normalizes the resulting document representations such that their vectors have length one.
- We can see this by computing the dot-product between a vector representation and itself.
- For example, for the first document in the collection we have:

In [ ]:
vec = vector_outputs[0]
vec.multiply(vec).sum()

To calculate the dot-product we multiplied the sparse vector by itself and then took the sum.

The fact that the vectors have length one (almost length of one due to approximations in the representation) means that the dot-product between vectors computes the cosine of the angle between them.
- The cosine of the angle between tf-idf vectors provides a value in the range [0,1], that is often used to measure the similarity between documents.
- Let's compute the similarity between the first two documents in the collection:

In [ ]:
vector_outputs[0].multiply(vector_outputs[1]).sum()

Here is the second document:

In [ ]:
print(output[1])

The similarity value is zero, which isn't surprising since most documents don't share vocabulary.

The average vocabulary size of a document in the collection is:

In [ ]:
nonzero_count = vector_outputs.count_nonzero()
doc_count = vector_outputs.get_shape()[0]

print(f"Average document vocabulary size: {nonzero_count/doc_count}")

The first two answers came from different categories:

In [ ]:
print(f"1st answer: {output[0]}")
print(f"2st answer: {output[1]}")

What if we try to find all the documents similar to the first one? Let's see:

In [ ]:
for i in range(len(output)):
  if vector_outputs[0].multiply(vector_outputs[i]).sum() >= 0.4:
    print(i, ': ', output[i])

### Searching the collection based on keywords

We could even use the same approach to compute the similarity between a search query and each of the documents in a collection in order to find the one that best matches with a query:

In [ ]:
import numpy as np

#query = 'pregnancy'
#query = 'heart attack'
query = 'PAP test'

query_vec = vectorizer.transform([query])[0]

index = np.argmax([query_vec.multiply(vector_outputs[i]).sum() for i in range(len(output))])
print(documents[index])
print("Similarity: ", query_vec.multiply(vector_outputs[index]).sum())

In [ ]:
vector_inputs = vectorizer.transform(input)

In [ ]:
similarity_matrix_io = np.dot(vector_inputs, vector_outputs.T)
similarity_matrix_ii = np.dot(vector_inputs, vector_inputs.T)
similarity_matrix_oo = np.dot(vector_outputs, vector_outputs.T)

In [ ]:
for i in range(similarity_matrix_io.shape[1]):
  if similarity_matrix_io[0, i] >= 0.4:
    print(i, ': ', output[i])

##Similarity Matrices

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
#@title question - answer similarity matrix
fig, ax = plt.subplots(figsize=(100, 100))

max_val = 100

intersection_matrix = similarity_matrix_io[:max_val, :max_val].todense()

ax.matshow(intersection_matrix, cmap=plt.cm.Blues)

for i in range(max_val):
    for j in range(max_val):
        c = np.round(intersection_matrix[j,i], 2)
        ax.text(i, j, str(c), va='center', ha='center')

In the similarity matrix generated from the initial 100 question-answer pairs, a notable trend emerges: there's a significant similarity between each question and its corresponding answer. This implies a close relationship between the question and its accurate response. Furthermore, some answers exhibit considerable similarity to different questions, suggesting that these questions may be about the same topics or fields.

In [ ]:
fig, ax = plt.subplots(figsize=(100, 100))
#@title question - question similarity matrix

max_val = 100

intersection_matrix = similarity_matrix_ii[:max_val, :max_val].todense()

ax.matshow(intersection_matrix, cmap=plt.cm.Blues)

for i in range(max_val):
    for j in range(max_val):
        c = np.round(intersection_matrix[j,i], 2)
        ax.text(i, j, str(c), va='center', ha='center')

In the similarity matrix derived from the initial 100 question-question pairs, it's notable that each question naturally shows high similarity with itself. However, the more interesting observation arises from the fact that without shuffling the dataset beforehand, similar questions consistently cluster together. This suggests that within the dataset, there are several questions per topic, and those related to the same topics tend to be grouped closely together.

In [ ]:
fig, ax = plt.subplots(figsize=(100, 100))

max_val = 100
#@title answer - answer similarity matrix


intersection_matrix = similarity_matrix_oo[:max_val, :max_val].todense()

ax.matshow(intersection_matrix, cmap=plt.cm.Blues)

for i in range(max_val):
    for j in range(max_val):
        c = np.round(intersection_matrix[j,i], 2)
        ax.text(i, j, str(c), va='center', ha='center')

In the similarity matrix derived from the initial 100 answer-answer pairs, a similar pattern emerges as observed in the analysis of the first 100 question-question pairs. However, this time, the clusters of answers with high similarity to themselves are even more pronounced.

# Clustering

Now we can start applying clustering to our vectorised documents

### Selecting a value for k

When used on low dimensional data, k-Means is often combined with the 'elbow method' (https://en.wikipedia.org/wiki/Elbow_method_(clustering)) for finding the 'right' number of clusters k

The method involves:
- running the clustering algorithm with increasing values of *k*
- plotting the intrinsic evaluation measure (within-cluster sum-of-squares)
- and looking for a point in which improvement in the measure decreases substantially from one time step to the next.

Let's try out the method using the MiniBatch version of k-Means since it is a bit faster to run.
- First generate the performance evaluation measure values across the range of k values:

In [ ]:
if True:
  performance_inertia = []
  performance_labels = []
  sum_time = 0
  k_values = range(10,200, 10)
  for k in k_values:
    print(f'Evaluation with k={k}')
    start =time.time()
    mini_batch = MiniBatchKMeans(n_clusters=k, batch_size=500, random_state=2307, n_init=2).fit(vector_outputs)
    end = time.time()
    execution_time = round(end-start, 2)
    sum_time += execution_time
    i = k_values.index(k)+1
    estimated_remaining_time = round((sum_time/i)*(len(k_values)-i), 2)
    performance_inertia.append(mini_batch.inertia_)
    performance_labels.append(mini_batch.labels_)
    print(f'Inertia:{mini_batch.inertia_}\nExecution time:{execution_time}s\nEstimated remaining time:{estimated_remaining_time}s\n\n')

Note that the within-cluster sum-of-squares almost always improves from one iteration to the next as k is increased.
- In theory it should always increase since the more cluster centroids there are, the more flexibility the model has for describing datapoints (assigning them to clusters)
- but in practice the stochasticity of k-Means and also the mini-batch procedure can cause the algorithm to not find the global optimum and thus produce a higher sum-of-squares value than a previous iteration (with lower k).

We'll now use some standard code to plot the performance measure against the value k:

In [ ]:
plt.figure()
plt.plot(k_values, performance_inertia)
plt.ylabel('Within-cluster sum-of-squares')
plt.xlabel('k')
plt.show()

Does it look to you like there is a point on the graph where performance suddenly stops getting a lot better?
- I don't see one ...
- That is likely because we have (i) very high dimensional data and (ii) quite a large number of documents.

So, we can try to analyze the silhouette score in the same range and look into the best scores to find a goof value of k

In [ ]:
from tqdm import tqdm
silhouette_scores = []
for labels in tqdm(performance_labels):
  silhouette_scores.append(metrics.silhouette_score(vector_outputs, labels))

In [ ]:
plt.figure()
plt.plot(k_values, silhouette_scores)
plt.ylabel('Silhouette_scores')
plt.xlabel('k')
plt.show()

In [ ]:
k = k_values[np.argmax(silhouette_scores)]
print(f'The value of K with the highest silhouette_scores is {k}')

### Clustering with k-Means

Now that we have computed a vector representation, we can perform the clustering.

There are many different clustering algorithms in common use, from k-Means and Hierarchical clustering to DBScan and Spectral clustering.  
- Many of them are implemented in Scikit-learn: https://scikit-learn.org/stable/modules/clustering.html
- You can easily change the code below to try them out.

We will use the k-Means algorithm since it is very popular, fast, scalable and relatively robust.
- k-Means models the dataset using k circlular clusters
- where each cluster is represented by the centroid of the datapoints it contains.

To apply k-Means we need to decide in advance how many clusters to look for.
- We will set the number of clusters to be exactly the number of categories in the dataset to see if the clustering can recover the original groups from the data.

In [ ]:
k = 10

kmeans = KMeans(n_clusters=k, max_iter=100, verbose=True, random_state=2307, n_init=2)
kmeans.fit(vector_outputs)


The inertia values are actually the *sum of squared distances between each sample to its closest cluster centroid*
- This is the measure that k-Means seeks to minimise
- Note: It is not the cosine distance between document and the cluster center (which is what we would like to minimise), but given that each vector has length one, the squared Euclidean distance is similar to it.  



### Clustering evaluation

Once we have run our clustering algorithm, we can

#### Investigating the Clusters

The clustering routine has now produced clusters in a very high dimensional feature space (with tens of thousands of dimansions).
- We can't plot such high dimensional data to see whether the clusters look coherent or not.
- Instead we will need to investigate the coherency of the clusters by looking at the terms occuring in them.

We can see the important terms for each cluster by inspecting the centroid vector for the cluster.
- Let's have a look at the terms with high weights in the centroid of the first cluster:

In [ ]:
# Get the centroid for the first cluster
centroid = kmeans.cluster_centers_[0]

# Sort terms according to their weights
# (argsort goes from lowest to highest, we reverse the order through slicing)
sorted_terms = centroid.argsort()[::-1]

# Print out the top 10 terms for the cluster
[vocab[j] for j in sorted_terms[:20]]

The top 10 terms look pretty consistent.

We could use our dot-product search trick from before to find the document in the collection that is the best exemplar of the cluster:

In [ ]:
index = np.argmax([np.dot(centroid,vec) for vec in vector_outputs.toarray()])

print(documents[index])

We now know what terms were important for defining cluster 0 and have seen a representative document,
- but still don't know how many documents were assigned to to the cluster.
- Let's find out:

In [ ]:
sum(kmeans.labels_ == 0)

We can see which clusters the first 10 documents have been assigned to:

In [ ]:
for i in range(10):
    print(f"document {i} is in cluster {kmeans.labels_[i]}")

Let's now repeat the process (algorithmically) to get the top terms for all the clusters.
- In the code below, we iterate over the clusters, getting the centroid for each, sort the values and printing out the top terms:

In [ ]:
print("Top terms per cluster:")
vocab = vectorizer.get_feature_names_out()

for i in range(kmeans.n_clusters):
    centroid = kmeans.cluster_centers_[i]
    sorted_terms = centroid.argsort()[::-1]
    print(f"Cluster {i}:\t{[vocab[j] for j in sorted_terms[:10]]}")

That's interesting.
- It looks like each of the clusters contains relatively consistent terms.

How many documents have been assigned to each cluster?

In [ ]:
print('Number of docs in: ')
number_of_elements = []
for i in range(kmeans.n_clusters):
    n_docs = np.sum(kmeans.labels_ == i)
    print(f"Cluster {i}: {n_docs}")
    number_of_elements.append(n_docs)

In [ ]:
plt.figure(figsize=(10, 5), dpi=125)
sns.histplot(number_of_elements, bins = k, kde=True)
plt.xlabel('Number of elemets')
plt.ylabel('Number of cluster')
plt.title('Number of elements per cluster')
plt.show()

Looks like there is a very large cluster corresponding to terms that might be more common across the corpus. Mostly, though, the counts are pretty well distributed across the clusters.

#### Quantitative Evaluation of Clustering Results

Let's now move on to the task of quantitatively evaluating the clustering algorithm. There are two ways to evaluate the results of a clustering algorithm:
- **Intrinsic evaluation** - If the ground truth is not known, you could use:
    - Within-cluster sum-of-squares: that is the *inertia* of the K-Means clustering
    - [Silhouette](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html#sklearn.metrics.silhouette_score) - "a higher Silhouette Coefficient score relates to a model with better defined clusters"
    
Let's first have a look at the *intrinsic measures*
- print out within-cluster sum-of-squares and the silhouette coefficient:


There are many possible metrics for clustering evaluation: some can be used when the ground truth labels are known, some can be used when the true labels are unknown.



In [ ]:
print("Intrinsic evaluation measures:")
print("Within-cluster sum-of-squares:", str(kmeans.inertia_))
print("Silhouette coefficient:", str(metrics.silhouette_score(vector_outputs, kmeans.labels_)))

The intrinsic evaluation measures are hard to interpret, since their values depend heavily on the difficulty of the clustering task, the amount of data, etc.
- Moreover, the sum-of-squares value will improve monotonically as the number of clusters is increased.


### Minibatch K-means clustering

MiniBatch k-Means is an approximate verions of the k-Means that is designed to scale up to massive datasets by making use of small samples (minibatches) in order to find the k centroids.
- It should be faster to run than k-Means. Let's give it a try:

In [ ]:
mb_kmeans = MiniBatchKMeans(n_clusters=k, batch_size=500, random_state=2307)
mb_kmeans.fit(vector_outputs)

Evaluate the minibatch clustering algorithm to see if the performance compares with the original k-Means on this data:

In [ ]:
print("Intrinsic evaluation measures:")
print("Within-cluster sum-of-squares:", str(mb_kmeans.inertia_))
print("Silhouette coefficient:", str(metrics.silhouette_score(vector_outputs, mb_kmeans.labels_)))

# Visualisation

Often visualisng the samples and the model output is useful to understand what's going on

### Problems visualising high dimensional data

For a bit of fun, we'll try to transform the high dimensional data into low dimensional data (just 3 dimensions) using a linear dimensionality reduction technique called Singular Value Decomposition.
- We'll transform the `vector_documents` to be 3 dimensional.

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(3)
reduced_data = svd.fit_transform(vector_outputs)

[x,y,z] = np.transpose(reduced_data)
[x,y,z]

We can plot the data coloured according to the clusters found by the original k-Means algorithm:

In [ ]:
fig = plt.figure(figsize=(15, 10))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x, y, z, c=kmeans.labels_, marker='.');

# Conclusion from clustering for Answers:

The clustering analysis reveals the main topics within the dataset, with terms consistently associated with specific medical themes, such as **vitamin deficiency**, **cancer types**, **thyroid conditions**, **heart issues**, **symptoms and treatments**, **kidney functions**, and **syndromes**.

Despite the comprehensive clustering, the **silhouette coefficient** is very **low (0.003886)**, indicating that the clusters might not be well-separated and suggesting a need for further refinement or exploration of other clustering techniques to achieve better-defined groupings.

The distribution of documents across clusters highlights varying concentrations, with some clusters (e.g., Cluster 6) containing significantly more documents than others, suggesting prevalent themes within the dataset.

**Term Frequency-Inverse Document Frequency (TF-IDF) Analysis:**
We analyzed the TF-IDF scores for terms within the clusters to identify the most significant terms. For example, terms such as **"levels"** and **"vitamin"** in **Cluster 0** have high TF-IDF scores, indicating their importance in the documents within that cluster. This approach helped us in understanding the key topics and terms that are characteristic of each cluster.

**Cluster Composition and Overlap:**
The cluster composition reveals that certain medical themes dominate the dataset. **Cluster 6**, with the highest number of documents **(15776)**, suggests a predominant focus on patient-related discussions and common diseases. On the other hand, clusters like **Cluster 2** (**330 documents**) focus on more specialized topics such as **thyroid conditions**, indicating less frequent but highly specific content.

**Evaluating Optimal Number of Clusters (k):**
We conducted an evaluation of MiniBatch K-Means clustering over a range of cluster numbers (k) from 10 to 190. The within-cluster sum-of-squares (inertia) decreases as k increases, but the silhouette coefficient remains low throughout. This suggests that while increasing the number of clusters might slightly improve cohesion within clusters, it does not necessarily enhance separation between them.

**Computational Performance:**
The computational performance analysis of MiniBatch K-Means clustering showed varying execution times with different values of k. The execution time generally increases with the number of clusters, reflecting the higher computational complexity. For example, k=10 took 0.44s, whereas k=190 took 17.44s. This information is crucial for optimizing the trade-off between clustering accuracy and computational efficiency.

#Context

#Clustering with contexts

In this section we have used the datasets which we generated using our context retrieval algorithm as the hugging face dataset did not have the context.

In [ ]:
file_dataframe = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP/NLP Project/Datasets/questions_context_dataframe.csv')
print(type(file_dataframe))

In [ ]:
file_dataframe.head()

In [ ]:
contexts = file_dataframe["contexts"]

##Context vectorization

We processesed a set of text contexts by first converting them into TF-IDF vectors using TfidfVectorizer.

Then, we evaluated the optimal number of clusters for K-Means clustering by iterating over different k values.

Futhermore, we computed the inertia (within-cluster sum-of-squares) and silhouette scores for each k value, aiding in the assessment of cluster quality.

Finally, we identified the k value with the highest silhouette score as the optimal number of clusters for the given text data.

In [ ]:
vectorizer = TfidfVectorizer(max_df=0.8, min_df=5, stop_words='english', )

In [ ]:
vectorizer.fit(contexts)

In [ ]:
vocab = vectorizer.get_feature_names_out()

print(f"Length of vocabulary: {len(vocab)}")

It's a lot bigger than using only the answers

In [ ]:
vector_contexts = vectorizer.transform(contexts)

In [ ]:
sorted([(vocab[j], vector_contexts[0, j]) for j in vector_contexts[0].nonzero()[1]], key=lambda x: -x[1])

We can try to compare the top words of the first context with the question and the answer to see if they are relevant

In [ ]:
print("Input: ", input[0], "\n")
print("Output: ", output[0], "\n")
print("Context: ", contexts[0])

Mg2, Ca2 and PTH are stricly connected with both the answer and the question

##Selecting a value of k

We evaluated the performance of MiniBatch K-Means clustering over a range of cluster numbers (k) from 10 to 190.

Futhermore, we measured and printed the inertia and execution time for each value of k and estimate the remaining time for the evaluations.

Finally, stored the results for further analysis to determine the optimal number of clusters.

In [ ]:
if True:
  performance_inertia = []
  performance_labels = []
  sum_time = 0
  k_values = range(10,200, 10)
  for k in k_values:
    print(f'Evaluation with k={k}')
    start =time.time()
    mini_batch = MiniBatchKMeans(n_clusters=k, batch_size=500, random_state=2307, n_init=2).fit(vector_contexts)
    end = time.time()
    execution_time = round(end-start, 2)
    sum_time += execution_time
    i = k_values.index(k)+1
    estimated_remaining_time = round((sum_time/i)*(len(k_values)-i), 2)
    performance_inertia.append(mini_batch.inertia_)
    performance_labels.append(mini_batch.labels_)
    print(f'Inertia:{mini_batch.inertia_}\nExecution time:{execution_time}s\nEstimated remaining time:{estimated_remaining_time}s\n\n')

In [ ]:
plt.figure()
plt.plot(k_values, performance_inertia)
plt.ylabel('Within-cluster sum-of-squares')
plt.xlabel('k')
plt.show()

From the graph it is impossible to understand the right value of k so we calculate the silhouette scores for the same k values

In [ ]:
from tqdm import tqdm
silhouette_scores = []
for labels in tqdm(performance_labels):
  silhouette_scores.append(metrics.silhouette_score(vector_contexts, labels))

In [ ]:
plt.figure()
plt.plot(k_values, silhouette_scores)
plt.ylabel('Silhouette_scores')
plt.xlabel('k')
plt.show()

In [ ]:
k = k_values[np.argmax(silhouette_scores)]
print(f'The value of K with the highest silhouette_scores is {k}')

##K-means clustering

We performed K-Means clustering to group similar text documents together. Here's what we learned from this analysis:

**Cluster Themes:** We identified 10 distinct groups (clusters) of text documents. Each cluster has a set of keywords that best represent the common themes or topics found within the documents in that group.

**Understanding Each Group:** We can see the top 10 most important words for each cluster. This helped us understand the core ideas that define each group of documents.

**Finding Representative Documents:** We found a document that closely resembles the central point (centroid) of the first cluster. This document served as a good example of the typical text found within that cluster.

**Distribution of Documents:** We explored how many documents belong to each cluster. We then visualized this distribution using a histogram. This helps us see how many documents fall into each group size category.

**Cluster Quality Check:** We performed some calculations to assess the quality of the clustering. These calculations tell us how tightly documents within each group resemble each other.

In short, this analysis helped us understand how the text documents can be grouped based on their content. We can see the key themes for each group, how many documents fall into each group, and how well the documents within each group are clustered together.

In [ ]:
k = 10

kmeans = KMeans(n_clusters=k, max_iter=100, verbose=True, random_state=2307, n_init=2)
kmeans.fit(vector_contexts)

In [ ]:
# Get the centroid for the first cluster
centroid = kmeans.cluster_centers_[0]

# Sort terms according to their weights
# (argsort goes from lowest to highest, we reverse the order through slicing)
sorted_terms = centroid.argsort()[::-1]

# Print out the top 10 terms for the cluster
[vocab[j] for j in sorted_terms[:20]]

In [ ]:
index = np.argmax([np.dot(centroid,vec) for vec in vector_contexts.toarray()])

print(contexts[index])

In [ ]:
print("Top terms per cluster:")
vocab = vectorizer.get_feature_names_out()

for i in range(kmeans.n_clusters):
    centroid = kmeans.cluster_centers_[i]
    sorted_terms = centroid.argsort()[::-1]
    print(f"Cluster {i}:\t{[vocab[j] for j in sorted_terms[:10]]}")

In [ ]:
print('Number of docs in: ')
number_of_elements = []
for i in range(kmeans.n_clusters):
    n_docs = np.sum(kmeans.labels_ == i)
    print(f"Cluster {i}: {n_docs}")
    number_of_elements.append(n_docs)

In [ ]:
plt.figure(figsize=(10, 5), dpi=125)
sns.histplot(number_of_elements, bins = k, kde=True)
plt.xlabel('Number of elemets')
plt.ylabel('Number of cluster')
plt.title('Number of elements per cluster')
plt.show()

In [ ]:
print("Intrinsic evaluation measures:")
print("Within-cluster sum-of-squares:", str(kmeans.inertia_))
print("Silhouette coefficient:", str(metrics.silhouette_score(vector_contexts, kmeans.labels_)))

# Conclusion from clustering for context:

Our clustering analysis reveals distinct themes in the dataset, including viral infections, case reports, syndromes, biological studies, treatments, cancer research, pulmonary diseases, epidemiological studies, cardiac conditions, and cellular biology.

The largest clusters are focused on biological studies and treatments, indicating a significant emphasis in these areas.


**Cluster Size Distribution:**



*   **Observation:** Cluster sizes vary significantly, with the largest cluster (Cluster 3) containing 12,542 documents and the smallest (Cluster 6) containing 742 documents.
*   **Implication:** This suggests that some topics are much more prevalent in the dataset. For instance, biological studies and treatments are heavily represented, indicating these are major areas of focus or interest in the dataset.

**Dominant Themes:**



*   **Clusters with Common Themes:** Multiple clusters (e.g., Clusters 1, 2, and 4) relate to patient care and treatment, though they differ in focus (case reports, clinical syndromes, and therapies).
*   **Implication:** There's a strong emphasis on clinical aspects of medicine, showing a potential overlap in topics, which might suggest a need for more granular clustering.

**Term Overlap:**

*   **Observation:** Some terms like "patients" and "case" appear in multiple clusters, indicating common topics across different clusters.
*   **Implication:** The dataset might benefit from a hierarchical clustering approach to better differentiate between overlapping themes.


**Intrinsic Evaluation Measures:**


*   **Within-cluster sum-of-squares (Inertia):** 32764.73
*   **Silhouette Coefficient:** 0.00296


The high inertia value suggests some clusters are not very compact, and the low silhouette coefficient indicates that the clusters are not well-separated.
This could be because we have a closed-domain dataset (medical one) covering a wide range of highly specilized topics.  To achieve a high level of internal cohesion within each cluster, we would need a large number of clusters, ensuring that each cluster contrains only closely related elements.


